In [ ]:
# import pandas as pd
# import numpy as np
# import os
# import  glob

# !pip install duckdb

# PHASE 1 — DATA FOUNDATION
- Let get all trade summary file from local folder
- merge them into single master csv data file
- normalize column names for better understanding and model standards

Company Name| Symbol| Share Volume| Trade Volume| Previous Close (Rs.)| Open (Rs.)| High (Rs.)|Low (Rs.)| **Last Trade (Rs.)| Change (Rs.)| Change (%)

##### Before modeling, you should now write a schema mapper that converts this raw exchange format → standardized ML format
Last Trade	    >  close<br>
Previous Close	>  prev_close<br>
Share Volume	>  volume<br>
Trade Volume	>  trades<br>

### Scan "C:\Users\Lapmart\Downloads\CSE" folder and crate master data set

## Data handling with pandas
- slower when having millions of data

In [ ]:
import pandas as pd
import os
import glob
from datetime import datetime


# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE"
OUTPUT = "MASTER_CSE_DATA.csv"

os.makedirs("log", exist_ok=True)
LOG_FILE = LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")


# ================ HELPERS =============

def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(msg + "\n")
    print(msg)

def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        log(f"### Error exctracting name from {filename}")
        return None

# Column mapping
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}


# ================ LOAD FILES ========================

files = glob.glob(os.path.join(FOLDER, "*.csv"))

log("=== LOADER START USING PANDAS FOR DATA ===")
log(f"Total files found: {len(files)}")

valid_frames = []
valid_dates = []

for file in files:
    name = os.path.basename(file)

     # ---- check filename date
    date = extract_date(name)
    if date is None:
        log(f"INVALID DATE FORMAT → skipped: {name}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
    except Exception as e:
        log(f"### READ ERROR → {name} → {e}")
        continue
        
    # ---- empty file check
    if df.empty:
        log(f"EMPTY FILE → skipped: {name}")
        continue

    # ---- normalize column names
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    # ---- add date column
    df["date"] = date

    valid_frames.append(df)
    valid_dates.append(date)


# ================ MERGE =======================

if not valid_frames:
    log("NO VALID FILES FOUND")
    exit()
    
# combine all files
master = pd.concat(valid_frames, ignore_index=True)


# ================ DUPLICATE CHECK =====================

dupes = master.duplicated(subset=["symbol","date"])
dup_count = dupes.sum()

if dup_count > 0:
    log(f"DUPLICATES FOUND: {dup_count} rows removed")
    master = master[~dupes]


# ================ SORT ========================

master = master.sort_values(["symbol", "date"])


# # ================ MISSING DATE REPORT ==================

# valid_dates = sorted(set(valid_dates))
# missing_days = []

# for i in range(len(valid_dates)-1):
#     diff = (valid_dates[i+1] - valid_dates[i]).days
#     if diff > 1:
#         missing_days.append((valid_dates[i], valid_dates[i+1]))

# if missing_days:
#     log("MISSING DATE GAPS:")
#     for g in missing_days:
#         log(f"{g[0].date()} → {g[1].date()}")


# ================ SAVE =======================
        
master.to_csv(OUTPUT, index=False)

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED ===")
print("DONE — Master dataset created using pandas")


## Data Hndling with DuckDB

### Why Data Engineers Prefer DuckDB
- Real data pipelines often use it for:
- stock market analysis
- log processing
- machine learning datasets
- financial backtesting
- Because it handles millions to billions of rows locally.

In [30]:
import pandas as pd
import os
import glob # finds files/folders matching patterns like wildcards in Python
from datetime import datetime
import duckdb



# ================ CONFIG ===================

FOLDER = r"C:\Users\Lapmart\Downloads\CSE" # my local stored folder path
OUTPUT = "MASTER_CSE_DATA.csv" # Name of the master data file

os.makedirs("log", exist_ok=True) # Create log folder if doesn't exit
# dynamically names a log file using the current date & time
LOG_FILE = os.path.join("log", f"loader_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")



# ================ HELPERS =================

# method to add log msg into log file & prints them to console
def log(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f: # "a" mode → appends messages instead of overwriting
        f.write(msg + "\n")
    print(msg)

# Method to extract date from file name in format like 260210_tradesummary.csv.
def extract_date(filename):
    try:
        raw = filename.split("_")[0]
        return datetime.strptime(raw, "%y%m%d")
    except:
        return None

# Standardize column names, helper map for mapping existing unusual column names into standard names 
COLUMN_MAP = {
    "Company Name": "company",
    "Symbol":"symbol",
    "Low (Rs.)": "low",
    "**Last Trade (Rs.)": "close",
    "Previous Close (Rs.)": "prev_close",
    "Open (Rs.)": "open",
    "High (Rs.)": "high",
    "Share Volume": "volume",
    "Trade Volume": "trades",
    "Change(Rs)": "change",
    "Change (Rs.)": "change",
    "Change (%)": "change_percentage"
}



# ================= CONNECT DUCKDB =================

con = duckdb.connect(database="cse_data.db")  # Create/connect DB
log("Creating DuckDB table...")

con.execute("""
CREATE TABLE IF NOT EXISTS stocks (
    company TEXT,
    symbol TEXT,
    low DOUBLE,
    close DOUBLE,
    prev_close DOUBLE,
    open DOUBLE,
    high DOUBLE,
    volume DOUBLE,
    trades DOUBLE,
    change DOUBLE,
    change_percentage DOUBLE,
    date DATE
)
""")



# ================= LOAD CSVs DIRECTLY =================

log("=== LOADER START WITH DUCKDB FOR DATA HANDLING ===")

files = glob.glob(os.path.join(FOLDER, "*.csv"))
log(f"Total files found: {len(files)}")

for file in files:
    filename = os.path.basename(file) # extract base name
    date = extract_date(filename) # get date from filename though function extract_date()
    
    if date is None:
        log(f"### INVALID DATE FORMAT → skipped: {filename}")
        continue

    # ---- read csv
    try:
        df = pd.read_csv(file)
        if df.empty:
            log(f"EMPTY FILE → skipped: {filename}")
            continue
    except Exception as e:
        log(f"READ ERROR → {filename} → {e}")
        continue
        
    # ---- Standardize column names, We assign a new list to df.columns to update the column names after processing them
    df.columns = [col.strip() for col in df.columns]  # remove extra spaces
    df.rename(columns=COLUMN_MAP, inplace=True) # Renames columns based on COLUMN_MAP.


    # ======== DATA QUALITY REPORT =========
    
    no_of_total_rows = len(df)
    no_of_missing_symbol = df["symbol"].isna().sum() if "symbol" in df.columns else total_rows
    no_of_missing_close = df["close"].isna().sum() if "close" in df.columns else total_rows

    numeric_cols = ["low","close","prev_close","open","high","volume","trades","change","change_percentage"]
    invalid_numeric = {}

    for col in numeric_cols:
        if col in df.columns:
            invalid_numeric[col] = pd.to_numeric(df[col], errors="coerce").isna().sum()
            
    if(no_of_missing_symbol > 0 or no_of_missing_close > 0):
        log("\n============== DATA QUALITY REPORT ==============")
        log(f"FILE: {filename}")
        log(f"Total rows: {total_rows}")
        log(f"Missing symbols: {missing_symbol}")
        log(f"Missing close prices: {missing_close}")
        log("============== ENF OF DATA QUALITY REPORT ==============\n")

    for col, count in invalid_numeric.items():
        if count > 0:
            log(f"Invalid numeric values in {col}: {count}")

    # check negative prices
    if "close" in df.columns:
        neg_price = (pd.to_numeric(df["close"], errors="coerce") <= 0).sum()
        if neg_price > 0:
            log(f"Invalid negative prices: {neg_price}")

    
    # ================= CLEAN DATA =================

    no_of_rows_before_clean = len(df)

    # remove rows where symbol missing BEFORE casting to string
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        
    # strip spaces
    if "symbol" in df.columns:
        df = df[df["symbol"].notna()]
        df["symbol"] = df["symbol"].astype(str).str.strip()

    if "company" in df.columns:
        df["company"] = df["company"].astype(str).str.strip()
    
    # convert numeric columns
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # remove rows where close price missing
    if "close" in df.columns:
        df = df[df["close"].notna()]

    # remove impossible values
    if "close" in df.columns:
        df = df[df["close"] > 0]

    if "volume" in df.columns:
        df = df[df["volume"] >= 0]

    # fix percentage anomalies
    if "change_percentage" in df.columns:
        df.loc[(df["change_percentage"] < -100) | (df["change_percentage"] > 1000), "change_percentage"] = None

    # drop fully empty rows
    df = df.dropna(how="all")

    no_of_rows_after_clean = len(df)
    
    if(no_of_rows_before_clean > no_of_rows_after_clean):
        log("\n========== DATA CLEAN REPORT ========")
        log(f"Some rows removed, File : {filename} before cleaning → {no_of_rows_before_clean} rows, After {no_of_rows_after_clean} rows\n")

    if len(df) == 0:
        log(f"ALL ROWS REMOVED AFTER CLEANING → skipped {filename}")
        continue
    
    # ---- add date column
    df["date"] = date

    # ------- Insert directly into DuckDB
    con.register("temp_df", df)
    con.execute("INSERT INTO stocks SELECT * FROM temp_df")
    # log(f"Inserted {len(df)} rows from {filename}")

log("All CSVs inserted into DuckDB.")



# ================  REMOVE DUPLICATES =====================
# Removes duplicate rows for the same symbol and date
# numbers duplicate rows starting from 1
# WHERE rn = 1 → keeps only the first row of each symbol/date combination

log("Removing duplicates...")

con.execute("""
CREATE OR REPLACE TABLE stocks_clean AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY symbol, date
               ORDER BY date
           ) AS rn
    FROM stocks
)
WHERE rn = 1
""")



# ================ SORT ========================

log("Sorting dataset...")

# Sorts the cleaned dataset by symbol then date
con.execute("""
CREATE OR REPLACE TABLE stocks_sorted AS
SELECT *
FROM stocks_clean
ORDER BY symbol, date
""")



# ================ CREATE INDEX ========================

log("Creating composite index on symbol+date for speed...")

# Adds a composite index on (symbol, date)
# Speeds up all queries that filter by symbol, date, or both
con.execute("""
CREATE INDEX IF NOT EXISTS idx_symbol_date ON stocks_sorted(symbol, date)
""")



# ================ MISSING DATE REPORT ==================
# TODO : if needed



# ================ EXPORT =======================

log("Exporting final CSV...")

con.execute(f"""
COPY stocks_sorted TO '{OUTPUT}' (HEADER, DELIMITER ',')
""")

log(f"MASTER DATASET CREATED → {OUTPUT}")
log("=== LOADER FINISHED USING DUCKDB ===")
print("DONE — Master dataset created")

Creating DuckDB table...
=== LOADER START WITH DUCKDB FOR DATA HANDLING ===
Total files found: 52

============== DATA QUALITY REPORT ==============
FILE: 260228_trade-summary-test.csv
Total rows: 293
Missing symbols: 1
Missing close prices: 1
============== ENF OF DATA QUALITY REPORT ==============

Invalid numeric values in close: 1

========== DATA CLEAN REPORT ========
Some rows removed, File : 260228_trade-summary-test.csv before cleaning → 293 rows, After 291 rows

All CSVs inserted into DuckDB.
Removing duplicates...
Sorting dataset...
Creating composite index on symbol+date for speed...
Exporting final CSV...
MASTER DATASET CREATED → MASTER_CSE_DATA.csv
=== LOADER FINISHED USING DUCKDB ===
DONE — Master dataset created


### Test speed of query execution with indexing

In [35]:
# commnet above code's indexing part and give a dofferent name for db cvvration try it with below code
# try agian with uncommenting

import time
import duckdb

# Change this to the DB you want to test
con = duckdb.connect("cse_data.db")  # or 'cse_data_index.db'

query = "SELECT * FROM stocks_sorted WHERE symbol='JKH'"

start = time.time()
results = con.execute(query).fetchall()
end = time.time()

print(f"Rows returned: {len(results)}")
print(f"Query time: {end - start:.4f} seconds")

Rows returned: 0
Query time: 0.0023 seconds


In [38]:
df.columns

Index(['company', 'symbol', 'volume', 'trades', 'prev_close', 'open', 'high',
       'low', 'close', 'change', 'change_percentage', 'date'],
      dtype='object')

In [39]:
df = pd.read_csv(r"C:\Users\Lapmart\Jupyter AI Models\CSE Investment Assistence/MASTER_CSE_DATA.csv")
df.head()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
0,ASIA ASSET FINANCE PLC,AAF.N0000,184992.0,102.0,61.7,61.5,63.0,60.8,60.9,-0.8,-1.30,2025-10-23,1
1,ASIA ASSET FINANCE PLC,AAF.N0000,225496.0,128.0,60.9,61.6,62.0,60.0,60.3,-0.6,-0.99,2025-10-24,1
2,ASIA ASSET FINANCE PLC,AAF.N0000,217100.0,171.0,60.3,61.0,61.0,58.5,59.2,-1.1,-1.82,2025-10-27,1
3,ASIA ASSET FINANCE PLC,AAF.N0000,107995.0,93.0,59.2,60.9,61.0,59.0,59.9,0.7,1.18,2025-10-28,1
4,ASIA ASSET FINANCE PLC,AAF.N0000,217448.0,101.0,59.9,60.0,62.0,59.0,60.5,0.6,1.00,2025-10-29,1


In [40]:
master_data_set.tail()

,company,symbol,low,close,prev_close,open,high,volume,trades,change,change_percentage,date,rn
14834,YORK ARCADE HOLDINGS PLC,YORK.N0000,219965.0,163.0,17.3,17.3,17.6,16.6,16.8,-0.5,-2.89,2026-02-23,1
14835,YORK ARCADE HOLDINGS PLC,YORK.N0000,131286.0,143.0,16.8,16.8,17.2,16.5,16.6,-0.2,-1.19,2026-02-24,1
14836,YORK ARCADE HOLDINGS PLC,YORK.N0000,240721.0,251.0,16.6,16.9,16.9,16.2,16.3,-0.3,-1.81,2026-02-25,1
14837,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-26,1
14838,YORK ARCADE HOLDINGS PLC,YORK.N0000,264415.0,211.0,16.3,16.3,16.4,16.0,16.1,-0.2,-1.23,2026-02-28,1


# PHASE 2 — FEATURE ENGINEERING ENGINE
- For 10 day profit

In [ ]:
# ============== Create Feature Table Base =====================
log("Creating feature base table...")

# Opens a connection to a DuckDB database file named cse_data.db, If it doesn’t exist, DuckDB creates it
con = duckdb.connect(database="cse_data.db")

# Creates a table called stocks_features, If it already exists → replaces it
# SELECT *,  > Selects all columns
con.execute("""
CREATE OR REPLACE TABLE stocks_features AS
SELECT
    *,
    
    LAG(close, 5)  OVER w AS close_5d,
    LAG(close, 10) OVER w AS close_10d,

    MAX(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS high_20,
    MIN(close) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS low_20,

    AVG(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS avg_vol_20,
    STDDEV(volume) OVER (w ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS std_vol_20,

    STDDEV(close) OVER (w ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) AS std_close_10

FROM stocks_sorted
WINDOW w AS (PARTITION BY symbol ORDER BY date);
""")

# =================== Creating Momentum Features ====================
log("Creating momentum features...")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_5 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_5 = (close / close_5d) - 1;
""")

con.execute("""
ALTER TABLE stocks_features ADD COLUMN ret_10 DOUBLE;
""")

con.execute("""
UPDATE stocks_features
SET ret_10 = (close / close_10d) - 1;
""")

# =================== Creating Liquidity Features ===================
log("Creating liquidity features...")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_ratio DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_ratio = volume / avg_vol_20;
""")

con.execute("ALTER TABLE stocks_features ADD COLUMN volume_z DOUBLE;")

con.execute("""
UPDATE stocks_features
SET volume_z = (volume - avg_vol_20) / std_vol_20;
""")

# =================== Creating Breakout Context ===================

con.execute("ALTER TABLE stocks_features ADD COLUMN range_position DOUBLE;")

con.execute("""
UPDATE stocks_features
SET range_position = (close - low_20) / (high_20 - low_20);
""")

# =================== Remove Invalid Rows ==============
log("Cleaning feature table...")

con.execute("""
CREATE OR REPLACE TABLE stocks_features_clean AS
SELECT *
FROM stocks_features
WHERE
    close_10d IS NOT NULL
    AND avg_vol_20 IS NOT NULL
    AND std_vol_20 IS NOT NULL
    AND std_close_10 IS NOT NULL
    AND volume > 0
    AND close > 0;
""")

# =================== Liquidity Ranking =====================
con.execute("ALTER TABLE stocks_features_clean ADD COLUMN liquidity_rank DOUBLE;")

con.execute("""
UPDATE stocks_features_clean t
SET liquidity_rank = r.rank
FROM (
    SELECT
        symbol,
        date,
        RANK() OVER (PARTITION BY date ORDER BY avg_vol_20 DESC) AS rank
    FROM stocks_features_clean
) r
WHERE t.symbol = r.symbol
AND t.date = r.date;
""")

# =================== Export Feature Dataset ==============
log("Exporting feature dataset...")

con.execute("""
COPY stocks_features_clean
TO 'features_master.csv'
(HEADER, DELIMITER ',');
""")


In [ ]:
print(con.execute("SELECT COUNT(*) FROM stocks_features_clean").fetchall())
print(con.execute("SELECT COUNT(*) FROM stocks_features_clean WHERE ret_10 IS NULL").fetchall())
print(con.execute("SELECT MIN(date), MAX(date) FROM stocks_features_clean").fetchall())